# 証券営業インテリジェンス ハンズオン
## Part 5: Cortex Agent（Snowflake Intelligence）

このノートブックでは、これまでに作成したすべてのサービスを統合した **Cortex Agent** を作成します。
作成したエージェントは **Snowflake Intelligence** の UI から自然言語で対話できます。

### Cortex Agent とは

**Cortex Analyst** + **Cortex Search** + **AI** を組み合わせた、マルチツール対話型AIアシスタントです。

```
ユーザーからの質問
      │
Cortex Agent（オーケストレーター）
  ├── 顧客データが必要 → Cortex Analyst（顧客資産DB）を呼び出し
  ├── ニュースが必要   → Cortex Search（NEWS_SEARCH_SERVICE）を呼び出し
  ├── 商品詳細が必要   → Cortex Search（LOAN_DOCS_SEARCH_SERVICE）を呼び出し
  └── 信託商品が必要   → Cortex Analyst（信託商品DB）を呼び出し
      │
  複数ツールの結果を統合して回答を生成
```

### 🚀 体験ポイント: データが増えるとAIが賢くなる！

| ステップ | 使用データ | AIの回答の質 |
|---|---|---|
| Step 1 | Cortex Search（ニュース・商品書・レポート）のみ | 市場情報は答えられるが、顧客データが見えない |
| Step 2 | + Cortex Analyst（顧客資産データ）を追加 | 「山田様のトヨタ株含み益は○○万円」と答えられる |
| Step 3 | + 信託商品データを追加 | 証券担保ローンの具体的な条件と税メリットまで提案 |

### 前提条件

- `part3_cortex_analyst.ipynb` 実行済み（Semantic View 2つ）
- `part4_cortex_search.ipynb` 実行済み（Cortex Search Service 3つ以上）

> ⏱️ **このパートの目安時間: 40分**


In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 前提条件の確認

Cortex Agent を作成する前に、必要なオブジェクトがすべて揃っているか確認します。

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_check_sv
-- Semantic View の確認
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_check_cs
-- Cortex Search Service の確認
-- READY ステータスになっていることを確認する
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## 2. 全ツール統合版エージェントの作成

Cortex Search と Cortex Analyst を統合した完全版エージェントを SQL で一括作成します。

> **💡 注意**: このセルを実行すると、エージェントが作成/更新されます。
> 「サンプル質問」の設定も含まれています。

> ⏱️ **目安: 5分**


In [ ]:
%%sql -r result_agent_full
-- ============================================================
-- Cortex Agent 完全版（5ツール + サンプル質問）
-- ============================================================
CREATE OR REPLACE AGENT WEALTH_MANAGEMENT_ASSISTANT_AGENT
WITH PROFILE = '{"display_name": "ウェルスマネジメントアシスタント"}'
COMMENT = '証券営業担当者向け総合支援AIアシスタント。顧客情報・ポートフォリオ分析・信託商品提案・市場ニュース・アナリストレポートを統合して最適な提案を支援します。'
FROM SPECIFICATION $$
{
    "models": {
        "orchestration": ""
    },
    "instructions": {
        "response": "あなたは証券営業担当者を支援するアシスタントです。\n\n【基本原則】\n・質問に直接回答し、求められていない情報は出さない\n・ツールで取得した情報のみを使用（推測禁止）\n・推奨は必ず前提条件付きで提示\n・不明点は「確認事項」として明示\n\n【出力形式】\nデフォルト：社内メモ（RM向け）\n顧客向けトーク：ユーザーが明示した場合のみ\n\n【個別顧客相談の構造】\n1. 確認事項：支払期限、取得単価、流動資産内訳、担保余力\n2. 現状サマリ：保有銘柄、時価、含み益\n3. 市況（必要時）：出典を明記\n4. 選択肢比較（表形式）\n5. 推奨（前提条件付き）＋リスク\n6. 次アクション\n\n【税金表示のルール】\n・取得単価不明時は税額を確定値で出さない\n・数値提示時は必ず：前提（税率/含み益）＋レンジ表示を付記\n\n【禁止事項】\n・取得単価未確認での税額確定値提示\n・必要資金に対する過剰売却・過剰借入の推奨",
        "orchestration": "【ツール選択原則】\n・質問に必要なツールのみ使用\n・PII（個人情報）は最小限、一覧では匿名化\n\n【ツール選択ルール】\n・顧客リスト/検索 → CUSTOMER_WEALTH_ANALYST\n・個別顧客照会 → CUSTOMER_WEALTH_ANALYST\n・ニュース照会 → NEWS_SEARCH\n・商品詳細条件 → LOAN_DOCS_SEARCH\n・信託商品を明示 → CUSTOMER_WEALTH_ANALYST + TRUST_PRODUCT_ANALYST\n・売却＋ニュース要望 → CUSTOMER_WEALTH_ANALYST + NEWS_SEARCH + ANALYST_REPORT_SEARCH"
    },
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "CUSTOMER_WEALTH_ANALYST",
                "description": "顧客データ（顧客マスタ・ポートフォリオ・取引履歴・相続税試算）を自然言語クエリで分析する。顧客検索・資産照会・含み益確認・相続税試算・ライフイベント確認に使用する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "TRUST_PRODUCT_ANALYST",
                "description": "信託銀行商品（証券担保ローン・教育資金贈与信託・遺言信託等）の条件・税制メリット・推奨条件を検索する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "NEWS_SEARCH",
                "description": "マーケットニュース50件を検索。銘柄・経済・税制・市場動向に関するニュースを取得する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "LOAN_DOCS_SEARCH",
                "description": "ローン・信託商品の説明書を検索。証券担保ローン・教育資金贈与信託・遺言信託などの詳細情報を取得する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "ANALYST_REPORT_SEARCH",
                "description": "アナリストレポートを検索。株式の投資判断・目標株価・リスク要因を取得する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "FUND_DOCS_SEARCH",
                "description": "野村AMファンドの目論見書を検索。ファンドの特色・投資リスク・信託報酬・分配金方針などの情報を取得する"
            }
        }
    ],
    "tool_resources": {
        "CUSTOMER_WEALTH_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW"
        },
        "TRUST_PRODUCT_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.TRUST_PRODUCT_SEMANTIC_VIEW"
        },
        "NEWS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "NEWS_ID"
        },
        "LOAN_DOCS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "DOC_ID"
        },
        "FUND_DOCS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "SECTION_HEADER",
            "id_column": "FILE_NAME"
        },
        "ANALYST_REPORT_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "REPORT_TITLE",
            "id_column": "REPORT_ID"
        }
    },
    "sample_questions": [
        "C001の山田太郎様のポートフォリオと含み益を教えてください",
        "証券担保ローンについて、条件と金利を教えてください",
        "トヨタ株の最新アナリスト評価と投資判断を教えてください",
        "相続税の試算が高い顧客を上位10名リストアップしてください",
        "教育資金贈与信託の制度について、期限も含めて教えてください"
    ]
}
$$;
SELECT 'Cortex Agent（完全版）作成完了！' AS STATUS;

In [ ]:
%%sql -r result_describe_agent
-- エージェントの詳細確認（追加されたツール一覧）
DESCRIBE AGENT SNOWFINANCE_DB.DEMO_SCHEMA.WEALTH_MANAGEMENT_ASSISTANT_AGENT;

## 3. Snowflake Intelligence への公開と動作確認

### Snowflake Intelligence へのアクセス手順

1. **Snowsight** にログイン
2. 左メニュー「**Snowflake Intelligence**」をクリック
3. 「**+ New Conversation**」をクリック
4. エージェント選択画面で `WEALTH_MANAGEMENT_ASSISTANT_AGENT` を選択
5. チャットを開始！

### まず基本動作を確認しましょう

```
C001の山田太郎様について教えてください
```

エージェントが複数ツールを呼び出して、顧客情報・ポートフォリオ・関連ニュースを統合した回答を返すことを確認します。

> ⏱️ **目安: 5分**（動作確認）


## 4. デモシナリオ: 山田様の相談

### 背景設定

```
顧客: 山田太郎様（68歳・元上場企業役員）
預かり資産: 約8億円
相談内容: 「孫の海外留学費用として2,000万円が必要。
           トヨタ株を売ろうかと思っている。」
```

---

### 🎯 Step 1: 顧客情報の確認

Snowflake Intelligence に以下を入力してみましょう：

```
C001の山田太郎様のプロフィールとポートフォリオを教えてください。
特にトヨタ株（7203）の保有状況と含み益を確認したいです。
```

**期待される回答**: 
- 山田様の基本情報（年齢・資産・担当RM）
- トヨタ株の保有株数・時価・含み益の金額
- 関連するライフイベント（孫の教育資金）

---

### 🎯 Step 2: 売却 vs ローンの比較提案

```
山田様がトヨタ株を売って2,000万円を確保しようとしています。
税金面でのデメリットと、証券担保ローンを使った場合との比較を教えてください。
```

**期待される回答（AIが複数ツールを同時使用）**:
- **Cortex Analyst**: 山田様のトヨタ株含み益 → 譲渡税の試算
- **LOAN_DOCS_SEARCH**: 証券担保ローンの金利・条件
- **NEWS_SEARCH**: トヨタ株の直近のアナリスト評価
- 比較表形式でのまとめ

---

### 🎯 Step 3: 長期的な相続・贈与対策

```
山田様の相続税の試算と、長期的な観点から教育資金贈与信託が有効かどうか教えてください。
制度の期限も含めてお願いします。
```

**期待される回答（さらに深い提案）**:
- **Cortex Analyst (INHERITANCE_TAX)**: 相続税試算額
- **TRUST_PRODUCT_ANALYST**: 教育資金贈与信託の条件・期限・税制メリット
- 短期対応（証券担保ローン）と長期対応（相続対策）の統合提案

---

### 🎯 Step 4: 訪問準備資料の自動作成

```
明日の山田様との訪問に向けて、提案書のドラフトを作成してください。
証券担保ローンと教育資金贈与信託の両方を盛り込んでください。
```

> ⏱️ **目安: 15分**（各質問をSnowflake Intelligenceに入力して体験）


## 5. よくある質問例（応用練習）

以下の質問を Snowflake Intelligence で試してみましょう。

### 顧客検索・分析
```
預かり資産5億円以上で、相続対策に関心がありそうな顧客をリストアップしてください

担当RM001の顧客で、最近30日間コンタクトしていない顧客を教えてください

トヨタ株を保有している顧客の含み益ランキングトップ10を出してください
```

### 銘柄分析・マーケット
```
トヨタ株の最新のアナリスト評価を教えてください

今週の市場ニュースで、特に重要なものを3つ教えてください

教育資金贈与信託の制度について、期限も含めて詳しく教えてください
```

### 相続税・試算
```
相続税が1億円を超える顧客は何人いますか？年齢帯別に集計してください

相続税対策として有効な信託商品はどれですか？条件も含めて教えてください
```

### マルチステップ推論（最も難しい質問）
```
来週の訪問リストに向けて、証券担保ローンの提案が特に有効な顧客を5名選んでください。
選定理由と各顧客への提案アプローチも添えてください。
```


## 6. エージェントのモニタリング

エージェントの動作を監視して、精度向上に役立てます。

### モニタリング手順

1. Snowsight → 「**AI と ML**」→「**エージェント**」を開く
2. `WEALTH_MANAGEMENT_ASSISTANT_AGENT` を選択
3. 「**分析**」タブで以下を確認：
   - **質問数・回答数**: 利用状況の把握
   - **ツール呼び出し頻度**: どのツールが多く使われているか
   - **レイテンシ**: 応答速度の監視
   - **Thumbs Up/Down**: ユーザーフィードバック

### パフォーマンス改善のポイント

- ⬆️ **Cortex Search の精度向上**: データ量を増やす、セクション分割を細かくする
- ⬆️ **Semantic View の充実**: 同義語・メトリクス定義を追加する
- ⬆️ **エージェント instruction の改善**: 回答形式・ツール選択ルールを調整


In [ ]:
%%sql -r result_final_tables
-- 作成されたすべてのオブジェクトの最終確認
SELECT 'TABLE' AS TYPE, TABLE_NAME AS OBJECT_NAME, '' AS STATUS
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TYPE, OBJECT_NAME;

In [ ]:
%%sql -r result_final_services
-- サービスとエージェントの確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_final_sv
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_final_agents
SHOW AGENTS IN SCHEMA DEMO_SCHEMA;

## まとめ

ハンズオン完了！4時間で証券営業インテリジェンスシステムを構築しました。

### 作成したオブジェクト一覧

| コンポーネント | オブジェクト名 | 役割 |
|---|---|---|
| **Cortex AI Functions** | AI_COMPLETE, AI_SENTIMENT, AI_EXTRACT, AI_CLASSIFY, AI_REDACT, AI_SIMILARITY | テキスト分析・生成 |
| **Cortex Analyst** | CUSTOMER_WEALTH_SEMANTIC_VIEW | 顧客資産データの自然言語クエリ |
| **Cortex Analyst** | TRUST_PRODUCT_SEMANTIC_VIEW | 信託商品の自然言語クエリ |
| **Cortex Search** | NEWS_SEARCH_SERVICE | ニュースのセマンティック検索 |
| **Cortex Search** | LOAN_DOCS_SEARCH_SERVICE | 商品説明書のセマンティック検索 |
| **Cortex Search** | ANALYST_REPORT_SEARCH_SERVICE | アナリストレポートのセマンティック検索 |
| **Cortex Agent** | WEALTH_MANAGEMENT_ASSISTANT_AGENT | 統合AIアシスタント |

### 4つの体験ポイント

1. **AI Functions**: 「2,000文字のレポートが3行に。1時間の準備が10秒に」
2. **Cortex Analyst**: 「SQLを書かずに日本語で顧客データを分析できる」
3. **Cortex Search**: 「証券担保ローンを知らなくても、意味を理解して発見できる」
4. **Cortex Agent**: 「データが増えるほど、AIの回答が賢くなる」

### 次のステップ

このハンズオンで作成したシステムを実際の業務データに適用する場合：

1. **実データの投入**: 顧客マスタ・取引履歴を実データに置き換え
2. **セキュリティ設定**: Row Access Policy で担当RM別にデータアクセス制限
3. **定期更新**: Cortex Search の TARGET_LAG をリアルタイムに近づける
4. **フィードバックループ**: エージェントのモニタリングで継続的改善

### ありがとうございました！

Snowflake Cortex AI と Snowflake Intelligence を活用することで、
**データドリブンな証券営業**が実現できます。
疑問点・ご要望はお気軽にお声がけください。
